# Notebook 06 — Application Integration & Demo Readiness

**Project:** CleanGanga-Prayagraj

This notebook is the integration layer between the completed analytical/AI notebooks and the final user-facing application.

It **does not repeat** EDA, hotspot calculations, Logistic Regression training, RAG construction, or RAG evaluation.

### Inputs from previous stages

- `data/prayagraj_assessment.csv` — cleaned assessment data from NB01
- `data/station_summary.csv` — station-level evidence from NB02
- `data/hotspot_ranking.csv` — ranked hotspot evidence from NB02
- `data/prototype_response.json` — prototype response artifact from NB05

### Goal

Build and validate a clean, UI-ready application payload so the eventual Streamlit/IBM Granite demo can use the existing project evidence without duplicating analytical logic.

## 1. Project architecture

```text
CPCB 2021 data
      ↓
NB01 — Data Quality & Assessment
      ↓
NB02 — Hotspot Analysis + ML Baseline
      ↓
NB03 — IBM Granite + RAG
      ↓
NB04 — RAG Evaluation + Responsible AI
      ↓
NB05 — Final Decision-Support Prototype
      ↓
NB06 — Application Integration
      ↓
Streamlit / Final Demo
```

The important engineering principle is **separation of responsibilities**:

- analysis produces evidence,
- RAG provides grounded knowledge,
- Granite explains evidence,
- the application presents the result.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

DATA_DIR = Path("../data")
EVALUATION_DIR = Path("../evaluation")

assessment_path = DATA_DIR / "prayagraj_assessment.csv"
station_path = DATA_DIR / "station_summary.csv"
hotspot_path = DATA_DIR / "hotspot_ranking.csv"
prototype_path = DATA_DIR / "prototype_response.json"

print("Data directory:", DATA_DIR.resolve())
print("Assessment:", assessment_path.exists())
print("Station summary:", station_path.exists())
print("Hotspot ranking:", hotspot_path.exists())
print("Prototype response:", prototype_path.exists())

Data directory: C:\Users\srich\OneDrive\Desktop\Technical\Projects\Internships\1M1B Virtual Internship\CleanGanga-Prayagraj\data
Assessment: True
Station summary: True
Hotspot ranking: True
Prototype response: True


## 2. Load the existing project artifacts

We load the outputs already produced by earlier notebooks.

Nothing is recomputed here.

In [2]:
required_files = {
    "assessment": assessment_path,
    "station_summary": station_path,
    "hotspot_ranking": hotspot_path
}

missing = [name for name, path in required_files.items() if not path.exists()]

if missing:
    raise FileNotFoundError(
        f"Missing required project artifacts: {missing}. "
        "Run the corresponding earlier notebook before continuing."
    )

assessment_df = pd.read_csv(assessment_path)
station_summary = pd.read_csv(station_path)
hotspot_ranking = pd.read_csv(hotspot_path)

prototype_response = None

if prototype_path.exists():
    with open(prototype_path, "r", encoding="utf-8") as f:
        prototype_response = json.load(f)

print("Assessment shape:", assessment_df.shape)
print("Station summary shape:", station_summary.shape)
print("Hotspot ranking shape:", hotspot_ranking.shape)
print("Prototype response loaded:", prototype_response is not None)

Assessment shape: (107, 25)
Station summary shape: (6, 28)
Hotspot ranking shape: (6, 28)
Prototype response loaded: True


## 3. Validate the application inputs

The application should fail clearly if an important field is unavailable.

We support the actual naming used by the project, including the NB02 baseline score `hotspot_score_baseline`.

In [3]:
required_station_columns = ["Station", "latitude", "longitude", "observations"]
missing_station_columns = [
    col for col in required_station_columns
    if col not in station_summary.columns
]

if missing_station_columns:
    raise KeyError(
        f"Missing station_summary columns: {missing_station_columns}"
    )

if "Station" not in hotspot_ranking.columns:
    raise KeyError("hotspot_ranking must contain a 'Station' column.")

score_column = None
for candidate in ["hotspot_score_baseline", "hotspot_score"]:
    if candidate in hotspot_ranking.columns:
        score_column = candidate
        break

if score_column is None:
    raise KeyError(
        "No hotspot score column found. Expected "
        "'hotspot_score_baseline' or 'hotspot_score'."
    )

print("Validation passed.")
print("Hotspot score column:", score_column)

Validation passed.
Hotspot score column: hotspot_score_baseline


## 4. Create a single station evidence record

The application needs one consistent structure for a selected station.

This function only **retrieves existing evidence**. It does not calculate a new pollution score.

In [4]:
def _value(row, candidates, default=None):
    for column in candidates:
        if column in row.index:
            value = row[column]
            if pd.notna(value):
                return value
    return default


def get_station_evidence(station_name):
    station_matches = station_summary[
        station_summary["Station"].astype(str).str.casefold()
        == str(station_name).casefold()
    ]

    if station_matches.empty:
        raise ValueError(f"Station not found: {station_name}")

    station_row = station_matches.iloc[0]

    hotspot_matches = hotspot_ranking[
        hotspot_ranking["Station"].astype(str).str.casefold()
        == str(station_name).casefold()
    ]

    hotspot_row = (
        hotspot_matches.iloc[0]
        if not hotspot_matches.empty
        else None
    )

    evidence = {
        "station": station_name,
        "latitude": _value(station_row, ["latitude", "Latitude"]),
        "longitude": _value(station_row, ["longitude", "Longitude"]),
        "observations": _value(station_row, ["observations"]),
        "persistence": _value(station_row, ["persistence"]),
        "mean_bod": _value(station_row, ["mean_bod"]),
        "max_bod": _value(station_row, ["max_bod"]),
        "mean_fc": _value(station_row, ["mean_fc"]),
        "max_fc": _value(station_row, ["max_fc"]),
        "anomaly_rate": _value(station_row, ["anomaly_rate"]),
        "hotspot_score": (
            _value(hotspot_row, [score_column])
            if hotspot_row is not None
            else None
        ),
        "hotspot_rank": (
            _value(hotspot_row, ["rank"])
            if hotspot_row is not None
            else None
        )
    }

    return evidence

## 5. Create the application payload

This is the main integration object.

It contains:

- selected station,
- measured/derived evidence,
- hotspot ranking,
- project limitations,
- responsible-AI disclaimer.

The future UI can consume this structure directly.

In [5]:
DISCLAIMER = (
    "This is a decision-support summary. "
    "Hotspot scores reflect 2021 CPCB data only and are not "
    "an official regulatory determination."
)


def build_application_payload(station_name, user_question):
    evidence = get_station_evidence(station_name)

    return {
        "project": "CleanGanga-Prayagraj",
        "station": evidence["station"],
        "user_question": user_question,
        "evidence": evidence,
        "knowledge_source_status": "Use verified RAG sources from NB03/NB05",
        "responsible_ai": {
            "no_fabricated_measurements": True,
            "no_unsupported_causation": True,
            "show_evidence": True,
            "show_uncertainty": True,
            "disclaimer": DISCLAIMER
        }
    }

## 6. Test the application payload

We test the highest-ranked station because it gives the demo a concrete example.

The values should come directly from the exported NB02 artifacts.

In [6]:
demo_station = hotspot_ranking.iloc[0]["Station"]

demo_payload = build_application_payload(
    demo_station,
    "Why is this station considered a potential pollution hotspot?"
)

print(json.dumps(demo_payload, indent=2, default=str))

{
  "project": "CleanGanga-Prayagraj",
  "station": "GANGA AT KADAGHAT ALLAHABAD",
  "user_question": "Why is this station considered a potential pollution hotspot?",
  "evidence": {
    "station": "GANGA AT KADAGHAT ALLAHABAD",
    "latitude": 25.443124,
    "longitude": 81.887148,
    "observations": "22",
    "persistence": 1.0,
    "mean_bod": 2.659090909090909,
    "max_bod": 2.9,
    "mean_fc": 960.9090909090908,
    "max_fc": 1400.0,
    "anomaly_rate": 0.0909090909090909,
    "hotspot_score": 0.8002239901135398,
    "hotspot_rank": null
  },
  "knowledge_source_status": "Use verified RAG sources from NB03/NB05",
  "responsible_ai": {
    "no_fabricated_measurements": true,
    "no_unsupported_causation": true,
    "show_evidence": true,
    "show_uncertainty": true,
    "disclaimer": "This is a decision-support summary. Hotspot scores reflect 2021 CPCB data only and are not an official regulatory determination."
  }
}


## 7. Build a dashboard summary

The application should also be able to show the overall situation before a user selects a station.

In [7]:
dashboard_summary = {
    "stations_monitored": int(hotspot_ranking["Station"].nunique()),
    "highest_ranked_station": str(hotspot_ranking.iloc[0]["Station"]),
    "highest_hotspot_score": float(
        pd.to_numeric(hotspot_ranking[score_column], errors="coerce").max()
    ),
    "assessment_observations": int(len(assessment_df)),
    "dataset_year": 2021
}

print(json.dumps(dashboard_summary, indent=2))

{
  "stations_monitored": 6,
  "highest_ranked_station": "GANGA AT KADAGHAT ALLAHABAD",
  "highest_hotspot_score": 0.8002239901135398,
  "assessment_observations": 107,
  "dataset_year": 2021
}


## 8. Prepare the final station table for the UI

This table is deliberately compact.

The UI does not need every internal column from the analytical notebooks.

In [8]:
ui_columns = [
    "Station",
    "latitude",
    "longitude",
    "observations",
    "persistence",
    "mean_bod",
    "mean_fc",
    "anomaly_rate"
]

available_ui_columns = [
    column for column in ui_columns
    if column in station_summary.columns
]

ui_station_table = station_summary[available_ui_columns].copy()

hotspot_ui = hotspot_ranking[["Station", score_column]].copy()

if "rank" in hotspot_ranking.columns:
    hotspot_ui["rank"] = hotspot_ranking["rank"]

ui_station_table = ui_station_table.merge(
    hotspot_ui,
    on="Station",
    how="left"
)

display(ui_station_table)

,Station,latitude,longitude,observations,persistence,mean_bod,mean_fc,anomaly_rate,hotspot_score_baseline
0,GANGA AT ALLAHABAD (RASOOLABAD) U.P.,25.502474,81.855439,23,1.000000,2.734783,1003.913043,0.043478,0.741279
1,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,25.419206,81.900522,23,1.000000,2.665217,1011.739130,0.043478,0.746377
2,GANGA AT KADAGHAT ALLAHABAD,25.443124,81.887148,22,1.000000,2.659091,960.909091,0.090909,0.800224
3,RIVER GANGA A/C TAMSA RIVER SIRSA SON BARSA,25.259363,82.096478,23,1.000000,2.621739,792.173913,0.043478,0.603358
4,TONS AT CHAKGHAT M.P.,25.043083,81.721004,5,0.000000,1.740000,2.000000,0.000000,0.000000
5,YAMUNA AT ALLAHABAD D/S (BALUA GHAT) U.P,25.422471,81.838806,11,0.818182,2.545455,654.545455,0.181818,0.710280


## 9. Export UI-ready artifacts

These files are integration artifacts, not replacements for the original analytical outputs.

They make the final application easier to build and test.

In [9]:
app_dir = DATA_DIR / "app"

app_dir.mkdir(exist_ok=True)

station_table_path = app_dir / "station_dashboard.csv"
demo_payload_path = app_dir / "demo_payload.json"
dashboard_path = app_dir / "dashboard_summary.json"

ui_station_table.to_csv(station_table_path, index=False)

with open(demo_payload_path, "w", encoding="utf-8") as f:
    json.dump(demo_payload, f, indent=2, default=str)

with open(dashboard_path, "w", encoding="utf-8") as f:
    json.dump(dashboard_summary, f, indent=2, default=str)

print("Saved:")
print("-", station_table_path)
print("-", demo_payload_path)
print("-", dashboard_path)

Saved:
- ..\data\app\station_dashboard.csv
- ..\data\app\demo_payload.json
- ..\data\app\dashboard_summary.json


## 10. End-to-end demo contract

The final application should follow this flow:

```text
User opens application
          ↓
Dashboard shows station ranking
          ↓
User selects a station
          ↓
Application loads existing station evidence
          ↓
User asks a question
          ↓
RAG retrieves verified knowledge
          ↓
IBM Granite explains the evidence
          ↓
UI displays:
   • station metrics
   • hotspot score/rank
   • Granite explanation
   • retrieved sources
   • limitations/disclaimer
```

**Important:** NB06 does not retrain the model or recalculate hotspot scores. It integrates the outputs already produced by the project.

## 11. Responsible AI checks before deployment

The final demo should satisfy all of these:

- [x] Measurements come from the project data.
- [x] Hotspot score comes from the deterministic NB02 analysis.
- [x] AI explanation is separated from computed evidence.
- [x] Unsupported pollution-source claims are prohibited.
- [x] The hotspot score is not presented as an official regulatory classification.
- [x] Dataset limitations are displayed.
- [x] API keys must remain outside Git.
- [x] Retrieved knowledge sources should be visible to the user.
- [x] Out-of-scope questions should receive a safe refusal rather than fabricated information.

# Conclusion

**Notebook 06 completes the integration layer.**

We now have:

```text
NB01  → Data preparation
NB02  → Hotspot analysis + ML baseline
NB03  → IBM Granite + RAG
NB04  → Evaluation + Responsible AI
NB05  → Decision-support prototype
NB06  → Application-ready integration artifacts
```

### Next major stage

Build the **Streamlit application** around the integration contract from this notebook.

The application should focus on usability and demonstration rather than introducing another analytical model.